# Logistic Regression

CSCI 6353 · Topic 22. Keep the linear score w·x+b, but pass it through a sigmoid to get a probability, and train with the convex cross-entropy (log-loss) instead of MSE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## The sigmoid squashing function

σ(z) = 1/(1+e^-z) maps any real number into (0, 1). In σ(a·x + b), **a** sets the steepness and **b** shifts the center (where the curve crosses 0.5).

In [ ]:
x = np.linspace(-8, 8, 200)
for a, b, lab in [(1, 0, "a=1, b=0"), (3, 0, "a=3 (steeper)"), (1, 3, "b=3 (shift)")]:
    plt.plot(x, sigmoid(a*x + b), label=lab)
plt.axhline(0.5, ls=":", color="gray"); plt.legend(); plt.xlabel("x"); plt.ylabel("σ(ax+b)")
plt.title("Sigmoid"); plt.grid(alpha=.3); plt.show()

## Why not least squares? MSE on the sigmoid is non-convex

Fixing b=1 and sweeping w, MSE on the *linear* model is a convex bowl, but MSE on the *sigmoid* is bumpy/flat, while the cross-entropy is convex again.

In [ ]:
x = np.array([1.3, 1.2, 3.5, 4.1]); y = np.array([0, 0, 1, 1]); b = 1
w = np.arange(-10, 10, 0.01)

mse_linear  = [np.mean((wi*x + b - y)**2)              for wi in w]
mse_sigmoid = [np.mean((sigmoid(wi*x + b) - y)**2)     for wi in w]
cross_ent   = [np.mean(-y*np.log(np.clip(sigmoid(wi*x+b),1e-9,1))
                       -(1-y)*np.log(np.clip(1-sigmoid(wi*x+b),1e-9,1))) for wi in w]

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
ax[0].plot(w, mse_linear);  ax[0].set_title("MSE on linear (convex)");   ax[0].set_ylim(0, 5)
ax[1].plot(w, mse_sigmoid); ax[1].set_title("MSE on sigmoid (non-convex)")
ax[2].plot(w, cross_ent);   ax[2].set_title("Cross-entropy (convex)")
for a in ax: a.set_xlabel("w"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## The log-loss (cross-entropy)

For one example: -log(h) when y=1, -log(1-h) when y=0. A confident-but-wrong prediction is punished toward infinity.

In [ ]:
h = np.linspace(0.001, 0.999, 200)
plt.plot(h, -np.log(h),   label="y=1: -log(h)")
plt.plot(h, -np.log(1-h), label="y=0: -log(1-h)")
plt.ylim(0, 6); plt.xlabel("predicted probability h"); plt.ylabel("loss")
plt.legend(); plt.grid(alpha=.3); plt.title("Log-loss"); plt.show()

## Train a 1D logistic regression

The gradient has the SAME form as linear regression: (1/n)·Σ(h - y)·x. The only change is h = sigmoid(w·x + b) instead of w·x + b.

In [ ]:
x = np.array([1.3, 1.2, 3.5, 4.1]); y = np.array([0, 0, 1, 1]); n = len(x)

w, b = 0.0, 0.0
alpha = 0.5
for _ in range(20000):
    h = sigmoid(w*x + b)                 # only change from linear regression
    w = w - alpha * (1/n) * np.sum((h - y) * x)
    b = b - alpha * (1/n) * np.sum( h - y)

print("w = %.4f  b = %.4f" % (w, b))     # -> 6.51  -15.39
print("decision boundary at x =", round(-b/w, 3))   # P = 0.5
for xt in [1.5, 3.0]:
    p = sigmoid(w*xt + b)
    print(f"x={xt}: P(y=1)={p:.3f} -> class {int(p >= 0.5)}")

In [ ]:
xs = np.linspace(0, 5.5, 300)
plt.plot(xs, sigmoid(w*xs + b), 'g', label="σ(w·x+b) = P(y=1)")
plt.scatter(x[y==0], y[y==0], color="blue", s=80, label="class 0")
plt.scatter(x[y==1], y[y==1], color="red",  s=80, label="class 1")
plt.axhline(0.5, ls=":", color="gray"); plt.axvline(-b/w, ls="--", color="k")
plt.xlabel("x"); plt.ylabel("probability"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Several features: same story, vectorized

The score is a dot product, the update is the vectorized gradient descent from multiple linear regression, with one sigmoid wrapped around it. Predict by thresholding the probability at 0.5.

In [ ]:
rng = np.random.default_rng(1)
A = rng.normal([1.5, 1.5], 0.6, (15, 2))
B = rng.normal([4.0, 4.0], 0.6, (15, 2))
X = np.vstack([A, B]); Y = np.array([0]*15 + [1]*15)

w = np.zeros(2); b = 0.0; alpha = 0.3
for _ in range(20000):
    h = sigmoid(X @ w + b)
    w = w - alpha * (1/len(X)) * (X.T @ (h - Y))
    b = b - alpha * (1/len(X)) * np.sum(h - Y)

pred = (sigmoid(X @ w + b) >= 0.5).astype(int)
print("w =", w.round(3), " b = %.3f" % b)
print("accuracy:", np.mean(pred == Y))    # -> 1.0